In [1]:
# ============================================================
# PHASE 4: DATA PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/telco_cleaned_v1.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [2]:
# ============================================================
# STEP 1: Fix data types and drop useless columns
# ============================================================

# TotalCharges still needs numeric conversion (from Phase 2)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# The 11 missing TotalCharges rows have tenure=0 (new customers)
# These customers haven't been billed yet — impute with 0
df['TotalCharges'].fillna(0, inplace=True)

# customerID is just an identifier — no predictive value
df.drop(columns=['customerID'], inplace=True)

# Drop Churn_Binary if it exists from Phase 3
if 'Churn_Binary' in df.columns:
    df.drop(columns=['Churn_Binary'], inplace=True)

print("Missing values after fix:", df.isnull().sum().sum())
print("Shape after dropping customerID:", df.shape)

Missing values after fix: 0
Shape after dropping customerID: (7043, 20)


In [3]:
# ============================================================
# STEP 2: Encode the target variable
# Yes → 1, No → 0
# ============================================================

df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Target distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.1f}%")

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate: 26.5%


In [4]:
# ============================================================
# STEP 3: Categorize columns — critical before building pipeline
# ============================================================

# Target
target = 'Churn'

# Already numeric
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Binary Yes/No columns → simple label encode (0/1)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'MultipleLines', 'OnlineSecurity',
               'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies']

# Multi-category columns → One-Hot Encode
ohe_cols = ['InternetService', 'Contract', 'PaymentMethod']

# SeniorCitizen is already 0/1 — no encoding needed
# We'll handle it separately

print("Numeric columns   :", numeric_cols)
print("Binary columns    :", len(binary_cols), "columns")
print("OHE columns       :", ohe_cols)

Numeric columns   : ['tenure', 'MonthlyCharges', 'TotalCharges']
Binary columns    : 12 columns
OHE columns       : ['InternetService', 'Contract', 'PaymentMethod']


In [5]:
# ============================================================
# STEP 4: Label encode binary Yes/No columns
# Why: These are already binary in meaning — 0/1 is enough
# ============================================================

le = LabelEncoder()

for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    
# Also encode gender explicitly for clarity
# Female=0, Male=1 (LabelEncoder is alphabetical)
print("Sample after binary encoding:")
print(df[binary_cols].head(3))

Sample after binary encoding:
   gender  Partner  Dependents  PhoneService  PaperlessBilling  MultipleLines  \
0       0        1           0             0                 1              1   
1       1        0           0             1                 0              0   
2       1        0           0             1                 1              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  StreamingTV  \
0               0             2                 0            0            0   
1               2             0                 2            0            0   
2               2             2                 0            0            0   

   StreamingMovies  
0                0  
1                0  
2                0  


In [6]:
# ============================================================
# STEP 5: One-Hot Encode multi-category columns
# Why: Models can't handle strings — we convert each category
#      into its own 0/1 column
# drop_first=True removes one column to avoid multicollinearity
# ============================================================

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print("Shape after One-Hot Encoding:", df.shape)
print("\nNew columns added:")
new_cols = [c for c in df.columns if any(c.startswith(o) for o in ohe_cols)]
print(new_cols)

Shape after One-Hot Encoding: (7043, 24)

New columns added:
['InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [7]:
# ============================================================
# STEP 6: Train-Test Split
# stratify=y ensures both splits maintain the 26% churn ratio
# random_state=42 makes results reproducible
# ============================================================

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # IMPORTANT for imbalanced datasets
)

print("Training set  :", X_train.shape)
print("Test set      :", X_test.shape)
print("\nChurn rate in train:", y_train.mean().round(3))
print("Churn rate in test :", y_test.mean().round(3))
print("\nGood: Both splits have similar churn rate — stratify worked!")

Training set  : (5634, 23)
Test set      : (1409, 23)

Churn rate in train: 0.265
Churn rate in test : 0.265

Good: Both splits have similar churn rate — stratify worked!


In [8]:
# ============================================================
# STEP 7: Scale numeric features
# Why: Logistic Regression and distance-based models are 
#      sensitive to feature scale.
#      tenure (0-72) vs MonthlyCharges (18-118) are very 
#      different ranges — scaling fixes this.
# CRITICAL RULE: Fit scaler on TRAIN only, transform both.
#                Never fit on test — that causes data leakage.
# ============================================================

scaler = StandardScaler()

# Fit ONLY on training data
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Transform test data using TRAIN statistics
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")
print("\nNumeric column stats after scaling (train):")
print(X_train[numeric_cols].describe().round(3))
print("\nMean ≈ 0 and Std ≈ 1 → scaling worked correctly.")

Scaling complete.

Numeric column stats after scaling (train):
         tenure  MonthlyCharges  TotalCharges
count  5634.000        5634.000      5634.000
mean     -0.000          -0.000         0.000
std       1.000           1.000         1.000
min      -1.322          -1.544        -1.009
25%      -0.956          -0.971        -0.832
50%      -0.142           0.185        -0.397
75%       0.916           0.832         0.674
max       1.608           1.786         2.802

Mean ≈ 0 and Std ≈ 1 → scaling worked correctly.


In [9]:
# ============================================================
# STEP 8: Save everything for modeling phase
# ============================================================

import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler for use in deployment later
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

print("Saved:")
print("  X_train.csv →", X_train.shape)
print("  X_test.csv  →", X_test.shape)
print("  scaler.pkl  → models/")
print("\nFinal feature count:", X_train.shape[1])

Saved:
  X_train.csv → (5634, 23)
  X_test.csv  → (1409, 23)
  scaler.pkl  → models/

Final feature count: 23


In [10]:
# ============================================================
# PREPROCESSING SUMMARY
# ============================================================

print("""
╔══════════════════════════════════════════════════════════╗
║           PREPROCESSING SUMMARY                          ║
╠══════════════════════════════════════════════════════════╣
║ 1. Fixed TotalCharges dtype (object → float)             ║
║ 2. Imputed 11 missing TotalCharges with 0                ║
║ 3. Dropped customerID (no predictive value)              ║
║ 4. Encoded target: Yes→1, No→0                           ║
║ 5. Label encoded 12 binary Yes/No columns                ║
║ 6. One-Hot encoded 3 multi-category columns              ║
║ 7. Stratified 80/20 train-test split                     ║
║ 8. StandardScaler on numeric features (no leakage)       ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║           PREPROCESSING SUMMARY                          ║
╠══════════════════════════════════════════════════════════╣
║ 1. Fixed TotalCharges dtype (object → float)             ║
║ 2. Imputed 11 missing TotalCharges with 0                ║
║ 3. Dropped customerID (no predictive value)              ║
║ 4. Encoded target: Yes→1, No→0                           ║
║ 5. Label encoded 12 binary Yes/No columns                ║
║ 6. One-Hot encoded 3 multi-category columns              ║
║ 7. Stratified 80/20 train-test split                     ║
║ 8. StandardScaler on numeric features (no leakage)       ║
╚══════════════════════════════════════════════════════════╝

